In [ ]:
# For now, we still need to manually add atomworks dependency
import sys;
modelhub_test = "/home/rohith/modelhub/release"
sys.path.append(f'{modelhub_test}/models/rfd3/src')
sys.path.append(f'{modelhub_test}/models/mpnn/src')
sys.path.append(f'{modelhub_test}/src')

from atomworks.io.utils.visualize import view

In [ ]:
# Running RFD3
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine
conf = RFD3InferenceConfig(
    ckpt_path='/projects/ml/aa_design/models/rfd3_latest_cleaned.ckpt',
    specification={
        'length': 10
    },
    diffusion_batch_size=2,
)
model = RFD3InferenceEngine(**conf)
outputs = model.run(
    inputs=None,
    out_dir=None,
    n_batches=1,
)

In [ ]:
for idx, data in outputs.items():
    print(f"Output type for batch {idx}: {type(data)}[0] = {type(data[0])}")
    print(f"Output atom_array: {data[0].atom_array}")
    atom_array = data[0].atom_array
view(atom_array)

In [ ]:
# Running RFD3
from mpnn.inference_engines.mpnn import MPNNInferenceEngine

#see defaults in mpnn.utils.inference.py : MPNN_GLOBAL_INFERENCE_DEFAULTS
conf = {
    ## protein_mpnn options
    #"model_type":"protein_mpnn",
    #"checkpoint_path":"/databases/mpnn/vanilla_model_weights/v_48_020.pt",
    ## ligand_mpnn options
    "model_type":"ligand_mpnn",
    "checkpoint_path":"/databases/mpnn/ligand_mpnn_model_weights/s25_r010_t300_p.pt",
    "is_legacy_weights": True,
    "out_directory": None,
    "write_structures": False,
    "write_fasta": False
}

# defaults in from mpnn.utils.inference import MPNN_PER_INPUT_INFERENCE_DEFAULTS
inputs = [
        {
            "batch_size": 10,
            "remove_waters":True
        }
    ]

model = MPNNInferenceEngine(**conf)

## update to use atom_arrays from previous cell
outputs = model.run(input_dicts=inputs, atom_arrays=[atom_array])

In [ ]:
from biotite.structure import get_residue_starts
from atomworks.constants import STANDARD_AA

for item in outputs:
    atom_array = item.atom_array
    res_starts = get_residue_starts(atom_array)
    protein_seq = []
    for res_name in atom_array.res_name[res_starts]:
        if res_name in STANDARD_AA:
            protein_seq.append(str(res_name))
    print(protein_seq)